In [1]:
from __future__ import annotations

import itertools
import warnings
from collections import defaultdict

import mne
import pickle
import pathlib
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt

mne.set_log_level("WARNING")

sns.set_theme(context='notebook', style='whitegrid')

In [2]:
raw_dataset = None
RAW_DATASET_NAME = pathlib.Path('data/raw_dataset.pkl')
EPOCH_DATASET_NAME = pathlib.Path('data/epoch_dataset.pkl')

EPOCH_TYPE = 'PinchLog'
EPOCH_TYPE_NOPINCH = 'NoPinchLog'
META_COL = 'PinchType'
CLASSES = ['PINCH', 'GAZE', 'WIZZARD_OF_OZ', 'PINCH_IGNORED']

def write_raw_dataset(raw_dataset: dict, path: pathlib.Path):
    with open(path, 'wb') as f:
        pickle.dump(raw_dataset, f)

def read_raw_dataset(path: pathlib.Path):
    data = None
    with open(path, 'rb') as f:
        data = pickle.load(f)
    return data

In [3]:
raw = read_raw_dataset(RAW_DATASET_NAME)
epochs = read_raw_dataset(EPOCH_DATASET_NAME)

In [ ]:
# region helper functions
def iter_sessions(d):
    """Yield (subject, session, value) over the nested dict, sorted."""
    for sub in sorted(d):
        for ses in sorted(d[sub]):
            yield sub, ses, d[sub][ses]


def get_pinchlog(epochs: mne.Epochs) -> mne.Epochs:
    """Return only the PinchLog epochs.

    Works whether 'PinchLog' is an event_id key, a prefix of hierarchical
    event ids ('PinchLog/pinch'), or a metadata column value.
    """
    if epochs.metadata is not None and "EpochType" in epochs.metadata:
        return epochs[f'EpochType == "{EPOCH_TYPE}"']
    keys = [k for k in epochs.event_id if EPOCH_TYPE in k]
    if not keys:
        raise KeyError(f"No {EPOCH_TYPE} events in {list(epochs.event_id)}")
    return epochs[keys]


def by_class(epochs: mne.Epochs, cls: str) -> mne.Epochs:
    """Select one PinchType level via the metadata query interface."""
    return epochs[f'{META_COL} == "{cls}"']


def stack_metadata(epochs_dict) -> pd.DataFrame:
    """Concatenate all PinchLog metadata into one long dataframe with keys."""
    rows = []
    for sub, ses, ep in iter_sessions(epochs_dict):
        ep = get_pinchlog(ep)
        md = ep.metadata.copy().reset_index(drop=True)
        md["subject"], md["session"] = sub, ses
        md["epoch_idx"] = np.arange(len(md))
        md["onset_s"] = ep.events[:, 0] / ep.info["sfreq"]
        rows.append(md)
    return pd.concat(rows, ignore_index=True)


def common_channels(epochs_dict) -> list[str]:
    """Intersection of channel names across every session (order preserved)."""
    sets = [set(ep.ch_names) for _, _, ep in iter_sessions(epochs_dict)]
    keep = set.intersection(*sets)
    first = next(iter_sessions(epochs_dict))[2]
    return [ch for ch in first.ch_names if ch in keep]

# end